In [ ]:
import h5py
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input,Dense)
from tensorflow.keras.callbacks import EarlyStopping

Dataset_files = [
    "H-H1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "H-H1_GWOSC_16KHZ_R1-1187008867-32.hdf5",
    "L-L1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "L-L1_GWOSC_16KHZ_R1-1187008867-32.hdf5",
    "V-V1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "V-V1_GWOSC_16KHZ_R1-1187008867-32.hdf5"
    "H-H1_GWOSC_4KHZ_R1-1187006835-4096.hdf5",
    "H-H1_GWOSC_16KHZ_R1-1187006835-4096.hdf5",
    "L-L1_GWOSC_4KHZ_R1-1187006835-4096.hdf5",
    "L-L1_GWOSC_16KHZ_R1-1187006835-4096.hdf5",
    "V-V1_GWOSC_4KHZ_R1-1187006835-4096.hdf5"
]

def load_strain(file_path):
    with h5py.File(file_path, "r") as f:
        strain = f["strain"]["Strain"][:]
    return strain

WINDOW_SIZE = 4096
STEP_SIZE = 2048

X = []

def generate_windows(signal):

    for i in range(
        0,
        len(signal) - WINDOW_SIZE,
        STEP_SIZE
    ):

        segment = signal[i:i + WINDOW_SIZE]

        X.append(segment)

for file in positive_files:
    signal = load_strain(file)
    generate_windows(signal)

for file in negative_files:
    signal = load_strain(file)
    generate_windows(signal)

X = np.array(X)

print("Dataset Shape:", X.shape)

scaler = StandardScaler()

X = scaler.fit_transform(X)

X_train, X_test = train_test_split(
    X,
    test_size=0.20,
    random_state=42
)

input_dim = X_train.shape[1]

encoding_dim = 128

input_layer = Input(shape=(input_dim,))

encoder = Dense(
    1024,
    activation='relu'
)(input_layer)

encoder = Dense(
    512,
    activation='relu'
)(encoder)

encoder = Dense(
    256,
    activation='relu'
)(encoder)

encoded = Dense(
    encoding_dim,
    activation='relu',
    name="Encoded_Features"
)(encoder)

decoder = Dense(
    256,
    activation='relu'
)(encoded)

decoder = Dense(
    512,
    activation='relu'
)(decoder)

decoder = Dense(
    1024,
    activation='relu'
)(decoder)

decoded = Dense(
    input_dim,
    activation='linear'
)(decoder)

autoencoder = Model(
    inputs=input_layer,
    outputs=decoded
)

encoder_model = Model(
    inputs=input_layer,
    outputs=encoded
)

autoencoder.compile(
    optimizer='adam',
    loss='mse'
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train,
    X_train,
    validation_data=(X_test, X_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

reconstructed = autoencoder.predict(X_test)

encoded_features = encoder_model.predict(X)

print("Encoded Shape:", encoded_features.shape)

autoencoder.save(
    "GW_Autoencoder.h5"
)

encoder_model.save(
    "GW_Encoder.h5"
)

print("Models Saved")